# Part 3: 하이브리드 검색 — BM25 + Ensemble Retriever

1. **이론**: 의미 검색(FAISS)만으로는 부족한 이유
2. **BM25**: 키워드 기반 검색의 원리와 장단점
3. **Ensemble + RRF**: 두 검색의 장점을 결합하는 방법
4. **데모**: 동일 질문에 대해 FAISS / BM25 / Ensemble 검색 결과를 나란히 비교

## 3-1. 의미 검색(FAISS)의 한계

Part 1에서 사용한 FAISS는 **의미(semantic) 검색**입니다.
- 질문과 문서를 각각 벡터로 변환하고, 코사인 유사도로 순위를 매깁니다.
- "영업이익"과 "수익성"처럼 **의미가 비슷한 표현**을 잘 매칭합니다.

하지만 한계가 있습니다:
- **정확한 키워드 매칭이 약함**: "DS 부문"이라는 정확한 키워드가 문서에 있어도, 의미 공간에서 가까운 다른 문서가 먼저 올 수 있습니다.
- **숫자·고유명사에 약함**: "16.4조원", "SK하이닉스" 같은 정확한 토큰 매칭은 키워드 검색이 유리합니다.
- **희귀 용어 문제**: 임베딩 모델이 학습하지 못한 용어(사업부명, 약어 등)는 벡터 표현이 부정확할 수 있습니다.

## 3-2. BM25: 키워드 기반 검색의 원리

**BM25**(Best Matching 25)는 TF-IDF를 개선한 키워드 검색 알고리즘입니다.

핵심 아이디어:
- **TF (Term Frequency)**: 질문의 키워드가 문서에 많이 등장할수록 점수가 높음
- **IDF (Inverse Document Frequency)**: 전체 문서에서 드문 키워드일수록 가중치가 높음
- **문서 길이 정규화**: 긴 문서가 불공정하게 유리하지 않도록 보정

**FAISS vs BM25 비교:**

| 항목 | FAISS (의미 검색) | BM25 (키워드 검색) |
|------|------------------|-------------------|
| 매칭 방식 | 벡터 코사인 유사도 | 키워드 빈도 기반 |
| 강점 | 유의어, 의미 유사 표현 | 정확한 키워드, 숫자, 고유명사 |
| 약점 | 정확한 토큰 매칭 | 의미적 유사성 이해 불가 |
| 예시 | "매출 실적" → "수익 현황" 매칭 가능 | "DS 부문" → 정확히 "DS 부문" 포함 문서 우선 |

두 검색의 장점을 결합하면 더 좋은 결과를 얻을 수 있습니다. 이것이 **하이브리드 검색**입니다.

In [ ]:
import pickle  # 객체 저장/불러오기
import time  # 시간 측정
from langchain_community.vectorstores import FAISS  # 벡터 저장소
from langchain_community.embeddings import HuggingFaceEmbeddings  # 임베딩 모델
from langchain_community.retrievers import BM25Retriever  # 키워드 기반 검색기
from langchain.retrievers import EnsembleRetriever  # 여러 검색기를 결합하는 검색기
from langchain_openai import ChatOpenAI  # OpenAI 채팅 모델
from langchain_core.messages import HumanMessage  # 사용자 메시지 객체
from dotenv import load_dotenv  # 환경변수 로드

load_dotenv()  # .env 파일 불러오기

with open("extracted_chunk_splits.pkl", "rb") as f:
    splits_data = pickle.load(f)

splits = splits_data["vlm"]
print(f"VLM 추출 청크 수: {len(splits)}")
print(f"청크 예시 (첫 번째):\n{splits[3].page_content[:200]}...")

VLM 추출 청크 수: 356
청크 예시 (첫 번째):
```markdown
## 삼성전자/SK하이닉스: 흔들리지 않는 편안함

### 25년 4분기는 SK하이닉스, 26년 1분기는 삼성전자가 앞설 전망

- **25년 4분기 SK하이닉스 영업이익 더 크다**
  - 25년 4분기 삼성전자 영업이익 20.1조원, SK하이닉스 영업이익 19.2조원. 삼성전자 메모리 사업부 영업이익은 17.2조원
  - 이견 전망...


In [ ]:
# 공통 설정
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatOpenAI(model="gpt-4o", temperature=0)
TOP_K = 5

RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

/var/folders/48/477vrv_n08b185fbd1ycjrqh0000gn/T/ipykernel_15646/2226361193.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


## 3-3. Retriever 3종 구축: FAISS, BM25, Ensemble

In [ ]:
# 1) FAISS Retriever (의미 검색)
vs = FAISS.from_documents(splits, embeddings)  # 문서 청크로 FAISS 벡터스토어 생성
faiss_retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})  # 유사도 기반 검색기 생성

# 2) BM25 Retriever (키워드 검색)
bm25_retriever = BM25Retriever.from_documents(documents=splits, k=TOP_K)  # 키워드 기반 BM25 검색기 생성

# 3) Ensemble Retriever (FAISS 50% + BM25 50%)
faiss_for_ens = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})  # 앙상블용 FAISS 검색기
bm25_for_ens = BM25Retriever.from_documents(documents=splits, k=TOP_K)  # 앙상블용 BM25 검색기
ensemble_retriever = EnsembleRetriever(
    retrievers=[faiss_for_ens, bm25_for_ens], weights=[0.5, 0.5]  # 두 검색기를 5:5 비율로 결합
)

print("Retriever 3종 구축 완료")  # 검색기 생성 완료 메시지

Retriever 3종 구축 완료


## 3-4. 검색 결과 비교: 동일 질문으로 FAISS vs BM25

같은 질문을 두 Retriever에 던져서 **어떤 문서가 검색되는지** 비교합니다.
핵심은 검색된 문서의 **내용**과 **순서**가 어떻게 다른지 관찰하는 것입니다.

In [ ]:
demo_question = "삼성전자 DS 부문 2025년 4분기 영업이익은?"

질문: 삼성전자 DS 부문 2025년 4분기 영업이익은?



In [ ]:
def display_docs(docs, title):
    print(f"\n{'='*70}")  # 구분선 출력
    print(f" {title} — 검색된 문서 Top-{len(docs)}")  # 제목과 문서 개수 출력
    print(f"{'='*70}")  # 구분선 출력
    for i, doc in enumerate(docs):  # 검색된 문서를 순서대로 반복
        snippet = doc.page_content[:150].replace('\n', ' ')  # 본문 앞 150자를 한 줄로 정리
        page = doc.metadata.get('page', '?')  # 메타데이터에서 페이지 번호 조회
        print(f"\n[{i+1}] (page {page}) {snippet}...")  # 문서 번호, 페이지, 내용 일부 출력
    print()  # 마지막 줄바꿈

In [25]:
# FAISS 검색 결과
faiss_docs = faiss_retriever.invoke(demo_question)
display_docs(faiss_docs, "FAISS (의미 검색)")


 FAISS (의미 검색) — 검색된 문서 Top-5

[1] (page 8) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이익은 3분기 대비 감소하였다. DS는 큰 폭의 가격 상승으로 2배 이상 증가했고, 디스플레이는 소형, 대형 물량 증가로 66.7% 증가, MX는 일부 부문 및 ASP 하락으로 47.6% 감소, VD/가전은 적자폭이 확대되었다. 사업부별로는  1) DS 사업부는 2025...

[2] (page 7) ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 32.9% 증가, VD/가전은 연말 성수기 효과로 6.1% 증가, MX 사업부는 물량은 소폭 감소, ASP 하락 영향이 커서 14.1% 감소, 디스플레이는 중소형, 대형 모두 물량이 개선된 영향으로 17.4% 증가했다. 사업부별로는:  1) **DS 사업부 매출액**은 2025년 3분...

[3] (page 6) ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 증가, VD/가전은 연말 성수기 효과로 개선되었으나 MX 사업부는 물량 감소와 가격 하락의 복합적 영향으로 부진했고, 디스플레이는 감소함. 범용 모두 물량이 개선된 영향으로 증가하였다. 삼성전자의 2025년 4분기 영업이익은...

[4] (page 6) #

In [7]:
# BM25 검색 결과
bm25_docs = bm25_retriever.invoke(demo_question)
display_docs(bm25_docs, "BM25 (키워드 검색)")


 BM25 (키워드 검색) — 검색된 문서 Top-5

[1] (page 5) ```markdown ## 차별화된 Data(Bit Growth/ASP)  ### 25.40 DRAM ASP 상승률은 삼성전자가 높고  2025년 4분기 DRAM Bit Growth는 삼성전자 +2%, SK하이닉스 +1%, 2025년 4분기 DRAM ASP는 삼성전자 ...

[2] (page 8) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이...

[3] (page 4) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스, 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 25년 4분기 삼성전자 영업이익 20.1조원, SK하이닉...

[4] (page 6) ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3...

[5] (page 11) ```markdown ## 2026년 1분기 매출액은 103.1조원으로 예상  삼성전자의 2026년 1분기 매출액은 2025년 4분기 대비 9.8% 증가한 103.1조원으로 예상한다. DS와 MX 사업부 매출이 증가하고, 디스플레이, VD/가전은 감소할 것으로 예상한다...



In [8]:
# 두 Retriever가 가져온 문서의 겹침(overlap) 분석
faiss_pages = set(d.metadata.get('page', '?') for d in faiss_docs)
bm25_pages = set(d.metadata.get('page', '?') for d in bm25_docs)
overlap = faiss_pages & bm25_pages
only_faiss = faiss_pages - bm25_pages
only_bm25 = bm25_pages - faiss_pages

print(f"FAISS가 가져온 페이지: {sorted(faiss_pages)}")
print(f"BM25가 가져온 페이지: {sorted(bm25_pages)}")
print(f"공통 페이지: {sorted(overlap)}")
print(f"FAISS에만 있는 페이지: {sorted(only_faiss)}")
print(f"BM25에만 있는 페이지: {sorted(only_bm25)}")
print(f"\n→ 두 Retriever는 서로 다른 관점으로 문서를 가져옵니다.")

FAISS가 가져온 페이지: [6, 7, 8, 12]
BM25가 가져온 페이지: [4, 5, 6, 8, 11]
공통 페이지: [6, 8]
FAISS에만 있는 페이지: [7, 12]
BM25에만 있는 페이지: [4, 5, 11]

→ 두 Retriever는 서로 다른 관점으로 문서를 가져옵니다.


## 3-5. Ensemble Retriever와 Reciprocal Rank Fusion (RRF)

**Ensemble Retriever**는 여러 Retriever의 결과를 **RRF(Reciprocal Rank Fusion)**로 결합합니다.

RRF 점수 공식:
```
RRF_score(d) = Σ  weight_i / (k + rank_i(d))
```
- `rank_i(d)`: i번째 Retriever에서 문서 d의 순위
- `k`: 상수 (기본값 60), 상위 순위에 과도한 가중치가 가는 것을 방지
- `weight_i`: 각 Retriever의 가중치 (여기서는 0.5 / 0.5)

**효과**: FAISS에서 2위, BM25에서 4위인 문서는 두 점수가 합산되어 최종 상위에 올라갑니다.
한쪽에서만 높은 순위인 문서보다 **양쪽 모두에서 관련성 있는 문서**가 우선됩니다.

## 예시

가중치를 동일하게 `0.5, 0.5`로 주고, `k=60`이라고 가정해보겠습니다.

### 문서 A
- FAISS: 2등
- BM25: 4등

```text
RRF(A) = 0.5 / (60 + 2) + 0.5 / (60 + 4)
       = 0.5 / 62 + 0.5 / 64
       ≈ 0.00806 + 0.00781
       ≈ 0.01587

### 문서 B
- FAISS: 1등
- BM25: 20등

RRF(B) = 0.5 / (60 + 1) + 0.5 / (60 + 20)
       = 0.5 / 61 + 0.5 / 80
       ≈ 0.00820 + 0.00625
       ≈ 0.01445

In [ ]:
# Ensemble 검색 결과
ensemble_docs = ensemble_retriever.invoke(demo_question)  # 앙상블 검색기로 질문과 관련된 문서 검색
display_docs(ensemble_docs, "Ensemble (FAISS + BM25)")  # 검색된 문서 목록 출력

ens_pages = set(d.metadata.get('page', '?') for d in ensemble_docs)  # 검색된 문서의 페이지 번호 추출
print(f"Ensemble이 가져온 페이지: {sorted(ens_pages)}")  # 페이지 번호 정렬 후 출력
print(f"→ FAISS와 BM25의 결과가 융합되어 양쪽의 장점을 모두 반영합니다.")  # 앙상블 검색 의미 설명


 Ensemble (FAISS + BM25) — 검색된 문서 Top-8

[1] (page 8) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은 20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원이다. DS, 디스플레이를 제외한 나머지 사업부 영업이...

[2] (page 6) ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3...

[3] (page 5) ```markdown ## 차별화된 Data(Bit Growth/ASP)  ### 25.40 DRAM ASP 상승률은 삼성전자가 높고  2025년 4분기 DRAM Bit Growth는 삼성전자 +2%, SK하이닉스 +1%, 2025년 4분기 DRAM ASP는 삼성전자 ...

[4] (page 7) ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대비 증가하였다. DS는 가격 상승으로 32.9% 증가,...

[5] (page 4) ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스, 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 25년 4분기 삼성전자 영업이익 20.1조원, SK하이닉...

[6] (page 6) ### 26년 1분기, DS부 주도  삼성전자의 2026년 1분기 매출액은 2025년 4분기 대비 9.8% 증가한 103.1조원으로 예상한다. DS와 MX 사업부 매출이 증가하고, 디스플레

## 3-6. RAG 답변으로 효과 확인

각 Retriever로 검색한 문서를 맥락으로 사용하여 LLM에 동일 질문을 던집니다.
검색 품질이 답변 정확도에 어떤 영향을 미치는지 확인합니다.

In [ ]:
demo_question = "삼성전자 DS 부문 2025년 4분기 영업이익은?"  # 데모용 질문
expected_answer = "16.4조원"  # 기대 정답

def run_rag(docs, retriever_name):
    context = "\n---\n".join(d.page_content for d in docs)  # 검색된 문서를 하나의 컨텍스트로 결합
    t0 = time.time()  # 시작 시간 기록
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])  # LLM에 질문 전달
    elapsed = time.time() - t0  # 응답 소요 시간 계산
    answer = msg.content if hasattr(msg, "content") else str(msg)  # 응답 텍스트 추출
    contains_answer = expected_answer in answer  # 정답 문자열 포함 여부 확인
    print(f"\n[{retriever_name}]")  # Retriever 이름 출력
    print(f"  답변: {answer[:200]}")  # 답변 일부 출력
    print(f"  정답 포함 여부: {'O' if contains_answer else 'X'} (정답: {expected_answer})")  # 정답 포함 여부 출력
    print(f"  응답 시간: {elapsed:.2f}s")  # 응답 시간 출력

run_rag(faiss_docs, "FAISS")  # FAISS 결과로 RAG 실행
run_rag(bm25_docs, "BM25")  # BM25 결과로 RAG 실행
run_rag(ensemble_docs, "Ensemble")  # Ensemble 결과로 RAG 실행


[FAISS]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 2025년 3분기 대비 134.4% 증가한 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.14s

[BM25]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 0.75s

[Ensemble]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 2025년 3분기 대비 134.4% 증가한 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.18s


## 3-7. 정리: 언제 어떤 Retriever를 쓸 것인가

| Retriever | 추천 상황 |
|-----------|----------|
| **FAISS (의미 검색)** | 질문이 자연어로 길고, 유의어/의역이 많을 때 |
| **BM25 (키워드 검색)** | 정확한 용어, 숫자, 코드명 등을 검색할 때 |
| **Ensemble (하이브리드)** | 두 가지 장점을 모두 활용하고 싶을 때 (일반적으로 권장) |

실무에서는 **Ensemble을 기본값**으로 사용하고, 도메인 특성에 따라 가중치(weights)를 조절하는 것이 일반적입니다.
- 법률/의학 등 용어가 중요한 도메인: BM25 가중치를 높임 (e.g., 0.3 / 0.7)
- 대화형 질의가 많은 경우: FAISS 가중치를 높임 (e.g., 0.7 / 0.3)

# Part 4: Multi-Query Retrieval — 질문 확장으로 검색 보완

1. **이론**: 단일 쿼리의 한계와 질문 확장의 필요성
2. **작동 방식**: LLM이 질문을 다양한 관점으로 변형하는 과정
3. **데모**: 확장된 쿼리들이 각각 어떤 문서를 가져오는지 확인
4. **노이즈 리스크**: 확장 질문이 원질문과 어긋나는 경우

## 4-1. 단일 쿼리의 한계

사용자는 질문을 **한 가지 표현**으로만 작성합니다. 하지만 같은 의도라도 표현은 다양할 수 있습니다:

| 원래 질문 | 같은 의도의 다른 표현 |
|-----------|---------------------|
| "삼성전자 DS 부문 영업이익은?" | "삼성 반도체 사업부 수익은?" |
| "SK하이닉스 목표주가는?" | "SK하이닉스 투자의견 및 적정가는?" |
| "누가 더 실적이 좋은가?" | "영업이익 비교", "실적 우위 기업" |

문서에 "반도체 사업부"로 기술되어 있는데 사용자가 "DS 부문"으로 물으면,
의미 검색이라도 최적의 문서를 놓칠 수 있습니다.

**Multi-Query Retrieval**은 LLM을 사용해 원질문을 여러 관점으로 변형(확장)하고,
각 변형 질문으로 검색한 결과를 합쳐서 이 문제를 해결합니다.

## 4-2. Multi-Query Retrieval 작동 방식

```
사용자 질문
    │
    ├── LLM이 3가지 변형 질문 생성
    │     ├── 변형 1 → 검색 → 문서 A, B, C
    │     ├── 변형 2 → 검색 → 문서 B, D, E
    │     └── 변형 3 → 검색 → 문서 A, E, F
    │
    └── 원본 질문 → 검색 → 문서 A, B, G
                      │
                      ▼
              합집합: A, B, C, D, E, F, G
              (중복 제거)
```

**장점**: 단일 질문으로 놓치기 쉬운 문서를 변형 질문이 보완

**비용**: LLM 호출 1회 추가 (질문 확장용, 가벼운 모델 사용 가능)

## 4-3. MultiQueryRetriever 구성

LangChain의 `MultiQueryRetriever`를 사용하면 질문 확장 → 각 질문별 검색 → 결과 합집합을 **자동으로** 처리합니다.

`from_llm()`으로 간단하게 생성할 수 있으며, 확장 질문 생성에는 가벼운 모델(`gpt-4o`)을 사용합니다.

In [ ]:
# 기본 Ensemble Retriever 구축 (Part 3에서 배운 하이브리드 검색)
vs = FAISS.from_documents(splits, embeddings)  # 문서 청크로 FAISS 벡터스토어 생성
faiss_r = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})  # 의미 기반 FAISS 검색기 생성
bm25_r = BM25Retriever.from_documents(documents=splits, k=TOP_K)  # 키워드 기반 BM25 검색기 생성
base_retriever = EnsembleRetriever(retrievers=[faiss_r, bm25_r], weights=[0.5, 0.5])  # FAISS와 BM25를 5:5로 결합한 기본 검색기

from langchain_core.prompts import PromptTemplate  # 프롬프트 템플릿 객체

multi_query_prompt = PromptTemplate(
    input_variables=["question"],  # 입력 변수로 question 사용
    template="""당신은 질문 확장 전문가입니다. 사용자의 질문을 여러 관점에서 변형하여 3가지 다른 질문을 만드세요.
각 질문은 서로 다른 관점이나 표현을 사용해야 합니다. 질문만 한 줄씩 출력하고, 번호나 기호 없이 질문 내용만 출력하세요.
원본 질문: {question}"""  # 원본 질문을 바탕으로 3개의 변형 질문 생성 지시
)

In [ ]:
from langchain.retrievers import MultiQueryRetriever, EnsembleRetriever  # 다중 질의 검색기와 앙상블 검색기

# MultiQueryRetriever 생성
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,  # 기본 검색기로 EnsembleRetriever 사용
    llm=ChatOpenAI(model="gpt-4o", temperature=0),  # 질문 확장용 LLM 설정
    include_original=True,  # 원본 질문도 함께 검색에 포함
)

print("MultiQueryRetriever 구성 완료")  # 생성 완료 메시지
print(f"  Base Retriever: EnsembleRetriever (FAISS + BM25)")  # 기본 검색기 설명
print(f"  Query 확장 LLM: gpt-4o")  # 질문 확장에 사용하는 모델 설명
print(f"  include_original=True → 원본 질문 검색 결과도 포함")  # 원본 질문 포함 여부 설명

MultiQueryRetriever 구성 완료
  Base Retriever: EnsembleRetriever (FAISS + BM25)
  Query 확장 LLM: gpt-4o
  include_original=True → 원본 질문 검색 결과도 포함


In [ ]:
demo_question = "2025년 4분기 DS 부문의 영업이익은?"  

# llm_chain으로 확장 질문 생성 (원본 + 변형 3개)
expanded = multi_retriever.llm_chain.invoke({"question": demo_question})  # LLM으로 질문 변형 생성
expanded_questions = [demo_question] + list(expanded)  # 원본 질문과 확장 질문들을 하나의 리스트로 결합

print("질문 확장 결과:")  
print("=" * 80)  
for i, q in enumerate(expanded_questions, 1):  # 질문들을 순서대로 반복
    print(f"\n[{i}] {q}")  # 번호와 함께 각 질문 출력

질문 확장 결과:

[1] 2025년 4분기 DS 부문의 영업이익은?

[2] - 2025년 4분기 DS 부문의 수익성은 어떻게 되나요?

[3] - 2025년 4분기 DS 부문의 재무 성과는 무엇인가요?

[4] - 2025년 4분기 DS 부문의 영업 실적은 어떠한가요?


In [ ]:
demo_question = "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?"

# llm_chain으로 확장 질문 생성 (원본 + 변형 3개)
expanded = multi_retriever.llm_chain.invoke({"question": demo_question}) # LLM으로 질문 변형 생성
expanded_questions = [demo_question] + list(expanded) # 원본 질문과 확장 질문들을 하나의 리스트로 결합

print("질문 확장 결과:")
print("=" * 80)
for i, q in enumerate(expanded_questions, 1): # 질문들을 순서대로 반복
    print(f"\n[{i}] {q}") # 번호와 함께 각 질문 출력

질문 확장 결과:

[1] 25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?

[2] - 2025년 4분기 삼성전자와 SK하이닉스의 영업이익을 비교하면 어느 회사가 더 높은가?

[3] - 삼성전자와 SK하이닉스의 2025년 4분기 영업이익 중 어느 회사가 더 많은 수익을 올렸는가?

[4] - 2025년 4분기 동안 삼성전자와 SK하이닉스 중 어느 회사의 영업이익이 더 컸는지 알고 싶습니다.


## 4-4. 확장 쿼리 생성 확인

`MultiQueryRetriever`를 호출하면 내부적으로 LLM이 질문을 확장합니다.
위에서 설정한 **로깅(DEBUG)**을 통해 어떤 쿼리들이 생성되었는지 직접 확인할 수 있습니다.

In [ ]:
demo_question = "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?"  

# MultiQueryRetriever invoke
multi_docs = multi_retriever.invoke(demo_question)  # 확장된 여러 질의로 문서 검색 수행
multi_pages = set(d.metadata.get('page', '?') for d in multi_docs)  # 검색된 문서의 페이지 번호 추출

print(f"\n검색된 문서 수: {len(multi_docs)}개")  # 검색된 전체 문서 수 출력
print(f"검색된 페이지: {sorted(multi_pages)}")  # 검색된 페이지 번호를 정렬해서 출력
print(f"\n상위 5개 문서 요약:\n" + "-"*40)  # 상위 문서 요약 제목 출력

for i, doc in enumerate(multi_docs[:5]):  # 상위 5개 문서만 반복
    page = doc.metadata.get('page', '?')  # 문서의 페이지 번호 조회
    snippet = doc.page_content.strip().replace('\n', ' ')  # 본문을 한 줄로 정리
    # 한 줄에 너무 길지 않게 60자씩 끊어서 보기 좋게 출력
    max_chars = 120  # 최대 표시 글자수
    wrapped = [snippet[j:j+60] for j in range(0, min(len(snippet), max_chars), 60)]  # 60자 단위로 분할
    print(f"[{i+1}] (page {page})")  # 문서 번호와 페이지 출력
    for w in wrapped:  # 잘라낸 각 줄을 순서대로 출력
        print(f"    {w}")
    print("-"*40)  # 문서 간 구분선 출력


검색된 문서 수: 9개
검색된 페이지: [4, 5, 8, 12, 17, 18, 20]

상위 5개 문서 요약:
----------------------------------------
[1] (page 4)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스
    , 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 2
----------------------------------------
[2] (page 17)
    ## SK하이닉스 상대주가 (%)  ![Graph](image.png)  ## SK하이닉스 (000660) 
     가파르지만 편하다.  ### 25년 4분기 예상치 상회  SK하이닉스의 2025년 4분기 매출액은 2025
----------------------------------------
[3] (page 12)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은
     35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 3
----------------------------------------
[4] (page 8)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은
     20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원
----------------------------------------
[5] (page 5)
    ```markdown ## 차별화된 Data(Bit Growth/ASP)  ### 25.40 DRAM ASP
     상승률은 삼성전자가 높고  2025년 4분기 DRAM Bit Growth는 삼성전자 +2%, SK하이닉스 
---------------

In [ ]:
# 비교: 기본 Retriever (질문 확장 없이)
original_docs = base_retriever.invoke(demo_question)  # 원본 질문만으로 기본 검색 수행
original_pages = set(d.metadata.get('page', '?') for d in original_docs)  # 기본 검색 결과의 페이지 번호 추출
new_pages = multi_pages - original_pages  # Multi-Query에서만 추가로 발견된 페이지 계산

print(f"[기본 Retriever]   문서 {len(original_docs)}개, 페이지: {sorted(original_pages)}")  # 기본 검색 결과 출력
print(f"[Multi-Query]      문서 {len(multi_docs)}개, 페이지: {sorted(multi_pages)}")  # Multi-Query 검색 결과 출력
print(f"\n→ Multi-Query를 통해 추가 발견된 페이지: {sorted(new_pages)} ({len(new_pages)}개)")  # 추가 발견 페이지 출력
print(f"  질문을 다양한 관점으로 확장함으로써 기본 검색이 놓친 문서를 보완합니다.")  # Multi-Query 효과 설명

[기본 Retriever]   문서 7개, 페이지: [4, 8, 12, 17, 18, 20]
[Multi-Query]      문서 9개, 페이지: [4, 5, 8, 12, 17, 18, 20]

→ Multi-Query를 통해 추가 발견된 페이지: [5] (1개)
  질문을 다양한 관점으로 확장함으로써 기본 검색이 놓친 문서를 보완합니다.


## 4-5. RAG 답변 비교: 원본 vs Multi-Query

검색 범위가 넓어지면 LLM 답변 품질이 어떻게 달라지는지 비교합니다.

In [ ]:
demo_question = "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?"  
expected_answer = "삼성전자"

def run_rag(docs, retriever_name):
    context = "\n---\n".join(d.page_content for d in docs)  # 검색된 문서를 하나의 컨텍스트로 결합
    t0 = time.time()  # 시작 시간 기록
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])  # LLM에 RAG 프롬프트 전달
    elapsed = time.time() - t0  # 응답 소요 시간 계산
    answer = msg.content if hasattr(msg, "content") else str(msg)  # 응답 텍스트 추출
    contains_answer = expected_answer in answer  # 정답 문자열 포함 여부 확인
    print(f"\n[{retriever_name}]")  # Retriever 이름 출력
    print(f"  답변: {answer[:300]}")  # 답변 일부 출력
    print(f"  정답 포함 여부: {'O' if contains_answer else 'X'} (정답: {expected_answer})")  # 정답 포함 여부 출력
    print(f"  컨텍스트 문서 수: {len(docs)}, 응답 시간: {elapsed:.2f}s")  # 문서 수와 응답 시간 출력

run_rag(original_docs, "Ensemble (원본 쿼리만)")  # 원본 질문으로만 검색한 결과로 RAG 실행
run_rag(multi_docs, "Multi-Query (확장 쿼리 포함)")  # 확장 질문까지 포함한 결과로 RAG 실행


[Ensemble (원본 쿼리만)]
  답변: 25년 4분기 영업이익은 삼성전자가 20.1조원으로 SK하이닉스의 19.2조원보다 더 큽니다.
  정답 포함 여부: O (정답: 삼성전자)
  컨텍스트 문서 수: 7, 응답 시간: 1.06s

[Multi-Query (확장 쿼리 포함)]
  답변: 25년 4분기 영업이익은 삼성전자가 20.1조원, SK하이닉스가 19.2조원으로, 삼성전자가 더 큽니다.
  정답 포함 여부: O (정답: 삼성전자)
  컨텍스트 문서 수: 9, 응답 시간: 1.31s


# Part 5: Cross-Encoder Rerank — 검색 결과 정밀 재정렬

1. **이론**: Bi-Encoder vs Cross-Encoder의 구조적 차이
2. **왜 Reranking이 필요한가**: 1차 검색의 한계
3. **데모**: Rerank 전후 문서 순위·점수 변화를 테이블로 시각화
4. **비용과 지연 시간**: 실측 데이터 확인

## 5-1. Bi-Encoder vs Cross-Encoder

RAG에서 사용하는 두 가지 유사도 모델 구조가 있습니다:

**Bi-Encoder (FAISS 등에서 사용)**
```
질문 → [Encoder] → 질문 벡터 ─┐
                                ├─ 코사인 유사도 → 점수
문서 → [Encoder] → 문서 벡터 ─┘
```
- 질문과 문서를 **독립적으로** 인코딩
- 매우 빠름 (문서 벡터를 미리 계산해두면 검색은 O(1))
- 질문-문서 간 **교차 어텐션**이 없어 세밀한 관련성 파악에 한계

**Cross-Encoder (Reranker에서 사용)**
```
질문 + 문서 → [Encoder] → 관련성 점수
```
- 질문과 문서를 **함께** 입력하여 교차 어텐션 수행
- 훨씬 정밀한 관련성 점수 산출
- 느림 (문서마다 별도 추론 필요 → top_k=20이면 20번 추론)

## 5-2. 왜 Reranking이 필요한가

1차 검색(FAISS/BM25/Ensemble)은 **속도 우선**으로 설계되어 있습니다.
넓은 범위에서 후보 문서(top_k=20)를 빠르게 가져오지만, 순위가 완벽하지 않습니다.

**2단계 검색 전략 (Retrieve → Rerank)**:
```
전체 문서 (수천~수만 개)
    │
    ├── 1단계: Retrieve (FAISS/BM25) → 후보 20개 (빠름)
    │
    └── 2단계: Rerank (Cross-Encoder) → 최종 5개 (정밀)
```

- 1단계에서 **재현율(Recall)** 확보: 관련 문서를 놓치지 않도록 넓게 검색
- 2단계에서 **정밀도(Precision)** 향상: 진짜 관련 있는 문서만 상위로
- LLM에 전달되는 맥락의 **품질**이 올라가므로 답변 정확도 향상

In [18]:
# Cross-Encoder 모델 로드
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('BAAI/bge-reranker-v2-m3')
print("bge-reranker-v2-m3 로드 완료")

bge-reranker-v2-m3 로드 완료


## 5-3. Rerank 데모: 검색 → 점수화 → 재정렬

 - 1단계로 Ensemble Retriever에서 20개 후보를 가져온 뒤,
 - Cross-Encoder로 각 문서의 관련성 점수를 매기고 재정렬합니다.
 - **Rerank 전후 순위가 어떻게 바뀌는지** 관찰합니다.

In [28]:
# 1단계: Ensemble Retriever로 후보 20개 검색
TOP_K = 20
TOP_N = 5

vs = FAISS.from_documents(splits, embeddings)  # 문서 청크로 FAISS 벡터스토어 생성
faiss_r = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})  # 의미 기반 FAISS 검색기 생성
bm25_r = BM25Retriever.from_documents(documents=splits, k=TOP_K)  # 키워드 기반 BM25 검색기 생성
retriever = EnsembleRetriever(retrievers=[faiss_r, bm25_r], weights=[0.5, 0.5])  # 두 검색기를 5:5로 결합

demo_question = "삼성전자 DS 부문 2025년 4분기 영업이익은?"  # 데모용 질문
candidate_docs = retriever.invoke(demo_question)  # 질문과 관련된 후보 문서 검색
print(f"1단계 검색 결과: {len(candidate_docs)}개 후보 문서")  # 검색된 후보 문서 수 출력

1단계 검색 결과: 25개 후보 문서


In [ ]:
# 2단계: Cross-Encoder로 각 문서에 관련성 점수 부여
t0 = time.time()  # 시작 시간 기록
pairs = [[demo_question, doc.page_content] for doc in candidate_docs]  # (질문, 문서내용) 쌍 생성
scores = cross_encoder.predict(pairs)  # 각 문서에 대해 관련성 점수 예측
rerank_time = time.time() - t0  # 재정렬 소요 시간 계산

scored_docs = list(zip(scores, range(len(candidate_docs)), candidate_docs))  # 점수, 기존 순위, 문서를 묶음
scored_docs_sorted = sorted(scored_docs, key=lambda x: x[0], reverse=True)  # 점수 기준 내림차순 정렬

print(f"Rerank 소요 시간: {rerank_time:.2f}s ({len(candidate_docs)}개 문서)")  # 리랭킹 시간 출력
print(f"\n{'='*80}")  # 구분선 출력
print(f" Rerank 전후 순위 변화 (상위 {TOP_N}개)")  # 제목 출력
print(f"{'='*80}")  # 구분선 출력
print(f"{'Rerank후':>8} {'Rerank전':>8} {'점수':>8}  {'내용 (앞 80자)'}")  # 표 헤더 출력
print(f"{'-'*8:>8} {'-'*8:>8} {'-'*8:>8}  {'-'*50}")  # 헤더 구분선 출력

for new_rank, (score, old_rank, doc) in enumerate(scored_docs_sorted[:TOP_N]):  # 상위 TOP_N개 문서만 반복
    page = doc.metadata.get('page', '?')  # 문서의 페이지 번호 조회
    snippet = doc.page_content.strip().replace('\n', ' ')  # 문서 본문을 한 줄로 정리
    max_chars = 120  # 최대 표시 글자수
    wrapped = [snippet[j:j+60] for j in range(0, min(len(snippet), max_chars), 60)]  # 60자씩 나눠 출력용 리스트 생성
    rank_change = old_rank - new_rank  # 기존 순위 대비 순위 변화 계산
    arrow = '↑' if rank_change > 0 else ('↓' if rank_change < 0 else '─')  # 순위 상승/하락/유지 표시
    print(f"  {new_rank+1:>4}위   {old_rank+1:>4}위  {score:>7.3f}  {arrow} (page {page})")  # 순위와 점수 출력
    for w in wrapped:  # 잘라낸 각 줄 출력
        print(f"       {w}")

Rerank 소요 시간: 2.34s (25개 문서)

 Rerank 전후 순위 변화 (상위 6개)
 Rerank후  Rerank전       점수  내용 (앞 80자)
-------- -------- --------  --------------------------------------------------
     1위      1위    1.000  ─ (page 8)
       ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은
        20.1조원  삼성전자의 2025년 4분기 영업이익은 2025년 3분기 대비 65.4% 증가한 20.1조원
     2위      5위    1.000  ↑ (page 7)
       ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2
       025년 3분기 대비 9.1% 증가한 93.8조원이다. MX/네트워크를 제외한 전 사업부 매출액은 3분기 대
     3위      2위    1.000  ↓ (page 6)
       ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, 
       압도적 메모리 실적  삼성전자의 2025년 4분기 매출액은 2025년 3분기 대비 9.1% 증가한 93.8조
     4위      4위    0.999  ─ (page 12)
       ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은
        35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 3
     5위      8위    0.998  ↑ (page 6)
       ### 26년 1분기, DS부 주도  삼성전자의 2026년 1분기 매출액은 2025년 4분기 대비 9.8% 
       증가한 103.1조원으로 예상한다. DS

In [31]:
# 전체 순위 변화 요약
print(f"\nRerank 전 Top-{TOP_N} 문서의 Rerank 후 순위:")  # 리랭킹 전 상위 문서들의 순위 변화 출력
for i, (score, old_rank, doc) in enumerate(scored_docs[:TOP_N]):  # 리랭킹 전 상위 TOP_N개 문서 반복
    # scored_docs의 old_rank == i, 즉 rerank전에 i위였던 문서
    # scored_docs_sorted에서 orig_idx == old_rank일 때의 new_rank를 찾는다
    new_rank = next(j for j, (_, orig_idx, _) in enumerate(scored_docs_sorted) if orig_idx == old_rank)  # 리랭킹 후 새 순위 찾기
    snippet = doc.page_content.strip().replace('\n', ' ')[:60]  # 문서 내용 앞 60자만 추출
    print(f"  Rerank 전 {i+1}위 → Rerank 후 {new_rank+1}위 (점수: {score:.3f}) | {snippet}...")  # 순위 변화 출력


Rerank 전 Top-5 문서의 Rerank 후 순위:
  Rerank 전 1위 → Rerank 후 1위 (점수: 1.000) | ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2025년 4분기 영업이익은...
  Rerank 전 2위 → Rerank 후 3위 (점수: 1.000) | ```markdown # 삼성전자 (005930)  ## 메모리가 끌고 가는 실적  ### 25년 4분기, ...
  Rerank 전 3위 → Rerank 후 6위 (점수: 0.993) | ```markdown ## 2026년 1분기 매출액은 103.1조원으로 예상  삼성전자의 2026년 1분기 ...
  Rerank 전 4위 → Rerank 후 4위 (점수: 0.999) | ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은...
  Rerank 전 5위 → Rerank 후 2위 (점수: 1.000) | ```markdown ## 2025년 4분기 매출액은 93.8조원  삼성전자의 2025년 4분기 매출액은 2...


#### Cross-Encoder가 질문과의 관련성을 정밀하게 재평가하여 순위가 변동됩니다.

## 5-4. top_n 파라미터와 정밀도-재현율 Trade-off

Rerank 후 **몇 개의 문서를 LLM에 전달할지** (top_n)는 중요한 하이퍼파라미터입니다:

| top_n | 정밀도 | 재현율 | LLM 비용 |
|-------|--------|--------|----------|
| 3 | 높음 | 낮을 수 있음 | 낮음 |
| 5 | 균형 | 균형 | 중간 |
| 10 | 낮을 수 있음 | 높음 | 높음 |

- **top_n이 작으면**: 정말 관련 있는 문서만 전달 → 정밀하지만 정보 부족 가능
- **top_n이 크면**: 많은 문서 전달 → 재현율은 높지만 노이즈도 증가, LLM 비용 증가

## 5-5. RAG 답변 비교: Rerank 적용 전 vs 후

In [ ]:
expected_answer = "16.4조원"  

# Rerank 전: 1단계 검색 결과에서 상위 5개만 사용
docs_before = candidate_docs[:TOP_N]  # 리랭킹 전 상위 문서 선택

# Rerank 후: Cross-Encoder 점수 기준 상위 5개 사용
docs_after = [doc for _, _, doc in scored_docs_sorted[:TOP_N]]  # 리랭킹 후 상위 문서 선택

def run_rag(docs, name):
    context = "\n---\n".join(d.page_content for d in docs)  # 검색된 문서를 하나의 컨텍스트로 결합
    t0 = time.time()  # 시작 시간 기록
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])  # LLM에 질문 전달
    elapsed = time.time() - t0  # 응답 소요 시간 계산
    answer = msg.content if hasattr(msg, "content") else str(msg)  # 응답 텍스트 추출
    contains = expected_answer in answer  # 정답 문자열 포함 여부 확인
    print(f"\n[{name}]")  # 실험 이름 출력
    print(f"  답변: {answer[:300]}")  # 답변 일부 출력
    print(f"  정답 포함 여부: {'O' if contains else 'X'} (정답: {expected_answer})")  # 정답 포함 여부 출력
    print(f"  응답 시간: {elapsed:.2f}s")  # 응답 시간 출력

run_rag(docs_before, "Rerank 전 (Ensemble Top-5)")  # 리랭킹 전 결과로 RAG 실행
run_rag(docs_after, "Rerank 후 (Cross-Encoder Top-5)")  # 리랭킹 후 결과로 RAG 실행


[Rerank 전 (Ensemble Top-5)]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.90s

[Rerank 후 (Cross-Encoder Top-5)]
  답변: 삼성전자 DS 부문의 2025년 4분기 영업이익은 16.4조원입니다.
  정답 포함 여부: O (정답: 16.4조원)
  응답 시간: 1.23s


## 5-6. 비용과 지연 시간

Cross-Encoder Reranking의 실제 비용을 정리합니다.

In [ ]:
# Rerank 시간 측정 (문서 수별)
for n in [5, 10, 20]:
    test_docs = candidate_docs[:n]  # 앞에서부터 n개 문서만 선택
    test_pairs = [[demo_question, d.page_content] for d in test_docs]  # (질문, 문서내용) 쌍 생성
    t0 = time.time()  # 시작 시간 기록
    _ = cross_encoder.predict(test_pairs)  # 선택한 문서들에 대해 리랭킹 점수 계산
    elapsed = time.time() - t0  # 소요 시간 계산
    print(f"문서 {n:>2}개 Rerank: {elapsed:.3f}s (문서당 {elapsed/n:.3f}s)")  # 전체/문서당 시간 출력

print(f"\n참고: bge-reranker-v2-m3는 로컬에서 실행되므로 API 비용은 없습니다.")  # 비용 안내

문서  5개 Rerank: 0.538s (문서당 0.108s)
문서 10개 Rerank: 0.891s (문서당 0.089s)
문서 20개 Rerank: 1.731s (문서당 0.087s)

참고: bge-reranker-v2-m3는 로컬에서 실행되므로 API 비용은 없습니다.


## 5-7. 정리

| 항목 | 설명 |
|------|------|
| **목적** | 1차 검색 결과를 정밀하게 재정렬하여 LLM에 전달할 맥락의 품질 향상 |
| **작동** | Cross-Encoder가 (질문, 문서) 쌍을 함께 입력받아 관련성 점수 산출 |
| **장점** | Bi-Encoder보다 훨씬 정밀한 관련성 판단, 정밀도(Precision) 향상 |
| **비용** | 문서 수만큼 Cross-Encoder 추론 필요 (CPU에서는 느림, GPU 권장) |
| **권장 설정** | top_k=20으로 넓게 검색 → Rerank → top_n=5로 압축 |

**전체 RAG 파이프라인 요약 (Part 2~5):**
```
PDF 추출 (Part 2)
  → 청킹 + 임베딩 인덱싱
    → 하이브리드 검색 (Part 3: FAISS + BM25)
      → 질문 확장 (Part 4: Multi-Query)
        → 정밀 재정렬 (Part 5: Cross-Encoder Rerank)
          → LLM 답변 생성
```
각 단계는 독립적으로 적용할 수도 있고, 조합하여 사용할 수도 있습니다.